In [7]:
import os
import json
from pathlib import Path
from datetime import datetime
import numpy as np

MODEL_TIER = "2b"
DRIVE_RESULTS_DIR = Path(f"/content/drive/MyDrive/vlm-finetuning-project1/results/baseline_test_full_{MODEL_TIER}")
JSONL_OUTPUT_PATH = str(DRIVE_RESULTS_DIR / "predictions.jsonl")

print(f"Target file: {JSONL_OUTPUT_PATH}")
assert os.path.exists(JSONL_OUTPUT_PATH), "File does not exist yet!"

file_stats = os.stat(JSONL_OUTPUT_PATH)
print(f"File Size: {file_stats.st_size / (1024*1024):.2f} MB")
print(f"Last Modified: {datetime.fromtimestamp(file_stats.st_mtime).strftime('%Y-%m-%d %H:%M:%S')}")

records = []
malformed_lines = []

with open(JSONL_OUTPUT_PATH, "r", encoding="utf-8") as f:
    for line_idx, line in enumerate(f):
        line = line.strip()
        if not line:
            continue
        try:
            rec = json.loads(line)
            records.append((line_idx, rec))
        except json.JSONDecodeError:
            malformed_lines.append(line_idx)

print(f"Total lines parsed: {len(records) + len(malformed_lines)}")
print(f"Valid JSON rows:    {len(records)}")
print(f"Malformed rows:     {len(malformed_lines)}  -> indices: {malformed_lines[:20]}{' ...' if len(malformed_lines)>20 else ''}")

failed_indices = []          # empty raw_output
missing_fence_indices = []   # non-empty but no ```json fence
success_indices = []         # non-empty AND fenced
latencies = []
image_id_by_idx = {}

for line_idx, rec in records:
    raw_output = rec.get("raw_output", "")
    image_id_by_idx[line_idx] = rec.get("image_id", "?")

    if not raw_output.strip():
        failed_indices.append(line_idx)
        continue

    if "```json" not in raw_output:
        missing_fence_indices.append(line_idx)
    else:
        success_indices.append(line_idx)

    lat = rec.get("latency_seconds", 0.0)
    if lat > 0:
        latencies.append(lat)

print(f"✅ Clean successes:         {len(success_indices)}")
print(f"⚠️ Missing ```json fence:   {len(missing_fence_indices)}")
print(f"❌ Empty (crashed) outputs: {len(failed_indices)}")


def find_continuous_ranges(indices_list):
    """Turns [5,6,7,20,21,50] into [(5,7), (20,21), (50,50)]"""
    if not indices_list:
        return []
    idxs = sorted(indices_list)
    ranges = []
    start = prev = idxs[0]
    for i in idxs[1:]:
        if i == prev + 1:
            prev = i
        else:
            ranges.append((start, prev))
            start = prev = i
    ranges.append((start, prev))
    return ranges

def print_ranges(name, indices_list, batch_size_guess=None):
    ranges = find_continuous_ranges(indices_list)
    print(f"\n{name}: {len(indices_list)} total, {len(ranges)} contiguous block(s)")
    for (s, e) in ranges:
        width = e - s + 1
        note = f" (~{width // batch_size_guess} batch(es) @ size {batch_size_guess})" if batch_size_guess else ""
        print(f"   [{s} -> {e}]  width={width}{note}")

print("=== FAILURE RANGE BREAKDOWN ===")
print_ranges("Empty/crashed records", failed_indices, batch_size_guess=32)
print_ranges("Malformed JSON lines", malformed_lines)
print_ranges("Missing ```json fence (successful but unformatted)", missing_fence_indices)

print("="*100)

total = len(records) + len(malformed_lines)
summary = {
    "total_lines": total,
    "malformed_json": len(malformed_lines),
    "empty_failed": len(failed_indices),
    "missing_fence": len(missing_fence_indices),
    "clean_success": len(success_indices),
    "success_rate_pct": round(len(success_indices) / total * 100, 2) if total else 0,
}
print("=== FINAL SUMMARY ===")
for k, v in summary.items():
    print(f"{k.ljust(20)}: {v}")

Target file: /content/drive/MyDrive/vlm-finetuning-project1/results/baseline_test_full_2b/predictions.jsonl
File Size: 4.95 MB
Last Modified: 2026-07-17 15:06:41
Total lines parsed: 3004
Valid JSON rows:    3004
Malformed rows:     0  -> indices: []
✅ Clean successes:         3004
⚠️ Missing ```json fence:   0
❌ Empty (crashed) outputs: 0
=== FAILURE RANGE BREAKDOWN ===

Empty/crashed records: 0 total, 0 contiguous block(s)

Malformed JSON lines: 0 total, 0 contiguous block(s)

Missing ```json fence (successful but unformatted): 0 total, 0 contiguous block(s)
=== FINAL SUMMARY ===
total_lines         : 3004
malformed_json      : 0
empty_failed        : 0
missing_fence       : 0
clean_success       : 3004
success_rate_pct    : 100.0


In [8]:
import os
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_from_disk

DATASET_PATH = "/content/drive/MyDrive/vlm-finetuning-project1/datasets/processed"
assert os.path.exists(DATASET_PATH), "Dataset not found at this path!"

print("Loading dataset...")
dataset = load_from_disk(DATASET_PATH)
print(dataset)

Loading dataset...
DatasetDict({
    train: Dataset({
        features: ['image', 'image_id', 'image_caption', 'illumination', 'camera_distance', 'view', 'quality_of_info', 'rule_1_violation', 'rule_2_violation', 'rule_3_violation', 'rule_4_violation', 'excavator', 'rebar', 'worker_with_white_hard_hat', 'resolution'],
        num_rows: 6308
    })
    val: Dataset({
        features: ['image', 'image_id', 'image_caption', 'illumination', 'camera_distance', 'view', 'quality_of_info', 'rule_1_violation', 'rule_2_violation', 'rule_3_violation', 'rule_4_violation', 'excavator', 'rebar', 'worker_with_white_hard_hat', 'resolution'],
        num_rows: 701
    })
    test: Dataset({
        features: ['image', 'image_id', 'image_caption', 'illumination', 'camera_distance', 'view', 'quality_of_info', 'rule_1_violation', 'rule_2_violation', 'rule_3_violation', 'rule_4_violation', 'excavator', 'rebar', 'worker_with_white_hard_hat', 'resolution'],
        num_rows: 3004
    })
})


In [10]:
all_stats = {}
PERCENTILES = [1, 5, 10, 25, 50, 75, 90, 95, 99]

for split in dataset.keys():
    res = np.array(dataset[split]["resolution"])

    # Pull width/height directly from the image objects (no .map, no schema change)
    widths = np.array([img.width for img in dataset[split]["image"]])
    heights = np.array([img.height for img in dataset[split]["image"]])
    aspect = widths / heights

    pcts = {f"p{p}": float(np.percentile(res, p)) for p in PERCENTILES}

    split_stats = {
        "count": len(res),
        "min_pixels": int(res.min()),
        "max_pixels": int(res.max()),
        "mean_pixels": float(res.mean()),
        "std_pixels": float(res.std()),
        "percentiles": pcts,
        "min_dims": f"{widths[res.argmin()]}x{heights[res.argmin()]}",
        "max_dims": f"{widths[res.argmax()]}x{heights[res.argmax()]}",
        "aspect_ratio_min": float(aspect.min()),
        "aspect_ratio_max": float(aspect.max()),
        "aspect_ratio_mean": float(aspect.mean()),
    }
    all_stats[split] = split_stats

    print(f"\n=== {split.upper()} ({len(res)} images) ===")
    print(f"Min:  {res.min():>12,} px  ({split_stats['min_dims']})")
    print(f"Max:  {res.max():>12,} px  ({split_stats['max_dims']})")
    print(f"Mean: {res.mean():>12,.0f} px   Std: {res.std():,.0f}")
    for p in PERCENTILES:
        print(f"  P{p:<3}: {pcts[f'p{p}']:>12,.0f} px")
    print(f"Aspect ratio range: {aspect.min():.2f} - {aspect.max():.2f} (mean {aspect.mean():.2f})")


=== TRAIN (6308 images) ===
Min:        47,232 px  (288x164)
Max:    14,625,792 px  (3312x4416)
Mean:    1,120,056 px   Std: 1,113,808
  P1  :      348,771 px
  P5  :      414,720 px
  P10 :      693,090 px
  P25 :      939,048 px
  P50 :    1,080,000 px
  P75 :    1,080,000 px
  P90 :    1,080,000 px
  P95 :    1,228,800 px
  P99 :    4,915,200 px
Aspect ratio range: 0.45 - 5.20 (mean 1.31)

=== VAL (701 images) ===
Min:       230,560 px  (524x440)
Max:    14,625,792 px  (4416x3312)
Mean:    1,216,896 px   Std: 1,506,937
  P1  :      388,800 px
  P5  :      414,720 px
  P10 :      693,090 px
  P25 :      843,936 px
  P50 :    1,080,000 px
  P75 :    1,080,000 px
  P90 :    1,080,000 px
  P95 :    1,555,200 px
  P99 :   11,808,768 px
Aspect ratio range: 0.56 - 4.55 (mean 1.32)

=== TEST (3004 images) ===
Min:        76,800 px  (320x240)
Max:    14,625,792 px  (4416x3312)
Mean:    1,157,066 px   Std: 1,132,906
  P1  :      388,800 px
  P5  :      688,892 px
  P10 :      750,000 px
  P2

In [11]:
BUCKETS = [
    (0, 100_000, "tiny (<100K px)"),
    (100_000, 500_000, "small (100K-500K)"),
    (500_000, 1_000_000, "medium (500K-1M)"),
    (1_000_000, 1_200_000, "standard (1M-1.2M) <- bulk of dataset"),
    (1_200_000, 3_000_000, "large (1.2M-3M)"),
    (3_000_000, 8_000_000, "very large (3M-8M)"),
    (8_000_000, float("inf"), "extreme outlier (8M+)"),
]

for split in dataset.keys():
    res = np.array(dataset[split]["resolution"])
    total = len(res)
    print(f"\n=== {split.upper()} BUCKET DISTRIBUTION ===")
    for lo, hi, label in BUCKETS:
        mask = (res >= lo) & (res < hi)
        count = mask.sum()
        pct = count / total * 100
        # find index range of this bucket ASSUMING the split is sorted by resolution
        idxs = np.where(mask)[0]
        idx_range = f"[{idxs.min()} to {idxs.max()}]" if len(idxs) else "none"
        print(f"  {label.ljust(38)}: {count:>5} imgs ({pct:5.1f}%)  index range {idx_range}")


=== TRAIN BUCKET DISTRIBUTION ===
  tiny (<100K px)                       :    13 imgs (  0.2%)  index range [0 to 12]
  small (100K-500K)                     :   344 imgs (  5.5%)  index range [13 to 356]
  medium (500K-1M)                      :  1570 imgs ( 24.9%)  index range [357 to 1926]
  standard (1M-1.2M) <- bulk of dataset :  3936 imgs ( 62.4%)  index range [1927 to 5862]
  large (1.2M-3M)                       :   315 imgs (  5.0%)  index range [5863 to 6177]
  very large (3M-8M)                    :    76 imgs (  1.2%)  index range [6178 to 6253]
  extreme outlier (8M+)                 :    54 imgs (  0.9%)  index range [6254 to 6307]

=== VAL BUCKET DISTRIBUTION ===
  tiny (<100K px)                       :     0 imgs (  0.0%)  index range none
  small (100K-500K)                     :    40 imgs (  5.7%)  index range [0 to 39]
  medium (500K-1M)                      :   186 imgs ( 26.5%)  index range [40 to 225]
  standard (1M-1.2M) <- bulk of dataset :   422 imgs ( 60.2

# Clean Corrupted/Empty Records from predictions.jsonl

In [12]:
import os
import json
import shutil
from pathlib import Path
from datetime import datetime

MODEL_TIER = "2b"
DRIVE_RESULTS_DIR = Path(f"/content/drive/MyDrive/vlm-finetuning-project1/results/baseline_test_full_{MODEL_TIER}")
JSONL_OUTPUT_PATH = str(DRIVE_RESULTS_DIR / "predictions.jsonl")

assert os.path.exists(JSONL_OUTPUT_PATH), "File not found!"

# Always back up before touching the file
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_path = str(DRIVE_RESULTS_DIR / f"predictions_backup_{timestamp}.jsonl")
shutil.copy(JSONL_OUTPUT_PATH, backup_path)
print(f"Backup created: {backup_path}")

Backup created: /content/drive/MyDrive/vlm-finetuning-project1/results/baseline_test_full_2b/predictions_backup_20260717_184017.jsonl


In [13]:
from datasets import load_from_disk

DATASET_PATH = "/content/drive/MyDrive/vlm-finetuning-project1/datasets/processed"
dataset = load_from_disk(DATASET_PATH)
EXPECTED_COUNT = len(dataset["test"])
print(f"Expected total records (test split size): {EXPECTED_COUNT}")

Expected total records (test split size): 3004


In [14]:
cleaned_records = []
seen_image_ids = set()
duplicates = []
empty_count = 0
malformed_count = 0

with open(JSONL_OUTPUT_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            record = json.loads(line)
        except json.JSONDecodeError:
            malformed_count += 1
            continue

        raw_output = record.get("raw_output", "")
        img_id = record.get("image_id")

        if not raw_output.strip():
            empty_count += 1
            continue

        if img_id in seen_image_ids:
            duplicates.append(img_id)
            continue  # keep the FIRST occurrence, drop later duplicates

        seen_image_ids.add(img_id)
        cleaned_records.append(record)

print(f"Malformed rows dropped:     {malformed_count}")
print(f"Empty/crashed rows dropped: {empty_count}")
print(f"Duplicate rows dropped:     {len(duplicates)}")
print(f"Clean unique records kept:  {len(cleaned_records)}")
print(f"Expected count:             {EXPECTED_COUNT}")

Malformed rows dropped:     0
Empty/crashed rows dropped: 0
Duplicate rows dropped:     0
Clean unique records kept:  3004
Expected count:             3004


In [15]:
all_expected_ids = set(str(x) for x in dataset["test"]["image_id"])
found_ids = set(str(x) for x in seen_image_ids)
missing_ids = sorted(all_expected_ids - found_ids)

print(f"Missing image_ids: {len(missing_ids)}")
if missing_ids:
    print(f"First 20 missing: {missing_ids[:20]}")

Missing image_ids: 0


In [ ]:
if len(cleaned_records) == EXPECTED_COUNT:
    print("Counts match exactly. Overwriting predictions.jsonl with cleaned data...")
    with open(JSONL_OUTPUT_PATH, "w", encoding="utf-8") as f:
        for record in cleaned_records:
            f.write(json.dumps(record) + "\n")
    print("✅ SUCCESS! predictions.jsonl is now clean, deduplicated, and ready for evaluation.")
    print(f"Backup remains at: {backup_path} (safe to delete once you confirm everything looks right)")
else:
    print(f"⚠️ Count mismatch: got {len(cleaned_records)}, expected {EXPECTED_COUNT}.")
    print("File was NOT overwritten. Options:")
    print("  1. Re-run inference for the missing_image_ids list, then re-run this task.")
    print("  2. If this is expected (e.g. --max_samples was used), manually confirm and overwrite yourself.")